In [3]:
import copy
import datetime as dt
from datetime import datetime
import importlib  # needed so that we can reload packages
import logging
import os
import pathlib
import sys
import time
import warnings
from typing import Union, Tuple

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from utils.logger_utils import setup_clean_logger, mute_external_loggers

# SISEPUEDE imports
from sisepuede.manager.sisepuede_examples import SISEPUEDEExamples
from sisepuede.manager.sisepuede_file_structure import SISEPUEDEFileStructure
import sisepuede.core.support_classes as sc
import sisepuede.utilities._plotting as spu
import sisepuede.utilities._toolbox as sf
import sisepuede.core.attribute_table as att
import sisepuede.manager.sisepuede_examples as sxl
import sisepuede.manager.sisepuede_file_structure as sfs
import sisepuede.manager.sisepuede_models as sm
import sisepuede.visualization.plots as svp



# --- Runtime configuration ---
warnings.filterwarnings("ignore")

# Set up a clean logger for your notebook
logger = setup_clean_logger("notebook", logging.INFO)
logger.info("Notebook started successfully.")

# Mute logs from sisepuede to avoid duplication
mute_external_loggers(["sisepuede"])
# run this on the terminal /opt/anaconda3/envs/ssp_libya_env_2/bin/python -m pip install "setuptools>=60,<70"

2026-09-17 16:42:59,728 - INFO - Notebook started successfully.


In [4]:
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


### Initial Set up

Make sure to edit the config yaml under ssp_modeling/config_files/config.yaml

You can also create a new config yaml



In [5]:
# Set up dir paths

CURR_DIR_PATH = pathlib.Path(os.getcwd())
SSP_MODELING_DIR_PATH = CURR_DIR_PATH.parent
PROJECT_DIR_PATH = SSP_MODELING_DIR_PATH.parent
DATA_DIR_PATH = SSP_MODELING_DIR_PATH.joinpath("input_data")
RUN_OUTPUT_DIR_PATH = SSP_MODELING_DIR_PATH.joinpath("ssp_run_output")
SCENARIO_MAPPING_DIR_PATH = SSP_MODELING_DIR_PATH.joinpath("scenario_mapping")
CONFIG_DIR_PATH = CURR_DIR_PATH.joinpath("config_files")
TRANSFORMATIONS_DIR_PATH = SSP_MODELING_DIR_PATH.joinpath("transformations")
MISC_DIR_PATH = SSP_MODELING_DIR_PATH.joinpath("misc")
STRATEGIES_DEFINITIONS_FILE_PATH = TRANSFORMATIONS_DIR_PATH.joinpath("strategy_definitions.csv")
STRATEGY_MAPPING_FILE_PATH = MISC_DIR_PATH.joinpath("strategy_mapping.yaml")

In [6]:
from ssp_transformations_handler.GeneralUtils import GeneralUtils
from ssp_transformations_handler.TransformationUtils import TransformationYamlProcessor, StrategyCSVHandler

# Initialize general utilities
g_utils = GeneralUtils()

In [7]:
# Load config file, double check your parameters are correct

YAML_FILE_PATH = os.path.join(CONFIG_DIR_PATH, "config.yaml")
config_params = g_utils.read_yaml(YAML_FILE_PATH)

country_name = config_params['country_name']
ssp_input_file_name = config_params['ssp_input_file_name']
ssp_transformation_cw = config_params['ssp_transformation_cw']
energy_model_flag = config_params['energy_model_flag']
set_lndu_reallocation_factor_to_zero_flag = config_params['set_lndu_reallocation_factor_to_zero']
sim_end_year = config_params.get('sim_end_year', 2050)  # Default to 2050 if not specified

# Print config parameters
logger.info(f"Country name: {country_name}")
logger.info(f"SSP input file name: {ssp_input_file_name}")
logger.info(f"SSP transformation CW: {ssp_transformation_cw}")
logger.info(f"Energy model flag: {energy_model_flag}")
logger.info(f"Set lndu reallocation factor to zero flag: {set_lndu_reallocation_factor_to_zero_flag}")
logger.info(f"Simulation end year: {sim_end_year}")

2026-09-17 16:42:59,794 - INFO - Country name: morocco
2026-09-17 16:42:59,795 - INFO - SSP input file name: sisepuede_raw_input_morocco_fuels_inen_ippu_scoe_prodcal_BLD_DEMAND_UP.csv
2026-09-17 16:42:59,795 - INFO - SSP transformation CW: ssp_morocco_transformation_cw.xlsx
2026-09-17 16:42:59,795 - INFO - Energy model flag: True
2026-09-17 16:42:59,795 - INFO - Set lndu reallocation factor to zero flag: True
2026-09-17 16:42:59,796 - INFO - Simulation end year: 2050


In [8]:
def get_file_structure(
    y0: int = 2015,
    y1: int = sim_end_year,
) -> Tuple[sfs.SISEPUEDEFileStructure, att.AttributeTable]:
    """Get the SISEPUEDE File Structure and update the attribute table
        with new years.
    """
    # setup some SISEPUEDE variables and update time period
    file_struct = sfs.SISEPUEDEFileStructure(
        initialize_directories = False,
    )
 
    # get some keys
    key_time_period = file_struct.model_attributes.dim_time_period
    key_year = file_struct.model_attributes.field_dim_year
 
 
    ##  BUILD THE ATTRIBUTE AND UPDATE
 
    # setup the new attribute table
    years = np.arange(y0, y1 + 1, ).astype(int)
    attribute_time_period = att.AttributeTable(
        pd.DataFrame(
            {
                key_time_period: range(len(years)),
                key_year: years,
            }
        ),
        key_time_period,
    )
 
    # finally, update the ModelAttributes inside the file structure
    (
        file_struct
        .model_attributes
        .update_dimensional_attribute_table(
            attribute_time_period,
        )
    )
 
    # return the tuple
    out = (file_struct, attribute_time_period, )
 
    return out


In [9]:
# Set up SSP objects
INPUT_FILE_PATH = DATA_DIR_PATH.joinpath(ssp_input_file_name)

# model attributes and associated support classes
_EXAMPLES = sxl.SISEPUEDEExamples()
_FILE_STRUCTURE, _ATTRIBUTE_TABLE_TIME_PERIOD = get_file_structure(y1=sim_end_year)
matt = _FILE_STRUCTURE.model_attributes
regions = sc.Regions(matt, )
time_periods = sc.TimePeriods(matt, )
 

In [10]:
# setup models in case we need them
models = sm.SISEPUEDEModels(
    matt,
    allow_electricity_run = True,
    fp_julia = _FILE_STRUCTURE.dir_jl,
    fp_nemomod_reference_files = _FILE_STRUCTURE.dir_ref_nemo,
    initialize_julia = False, 
)

### Making sure our input file has the correct format and correct columns
We use an example df with the complete fields and correct format to make sure our file is in the right shape

In [11]:
##  BUILD BASE INPUTS
df_inputs_raw = pd.read_csv(INPUT_FILE_PATH)

# pull example data to fill in gaps
df_example_input = _EXAMPLES("input_data_frame")

In [12]:
# Double checking that our df is in the correct shape (Empty sets should be printed to make sure everything is Ok!)
g_utils.compare_dfs(df_example_input, df_inputs_raw)

Columns in df_example but not in df_input: {'frac_lndu_seasonal_wetland_pastures', 'frac_lndu_seasonal_wetland_shrublands', 'frac_frst_forest_conversions_available_for_fuelwood', 'region', 'frac_lndu_seasonal_wetland_croplands', 'frac_lndu_seasonal_wetland_other', 'frac_trns_fuelmix_water_borne_gasoline', 'frac_lndu_seasonal_wetland_grasslands'}
Columns in df_input but not in df_example: {'frac_fgtv_capture_associated_gas_fuel_crude', 'ef_fgtv_distribution_tonne_co2_per_m3_fuel_crude', 'frac_fgtv_capture_associated_gas_fuel_coal', 'iso_alpha_3', 'frac_fgtv_capture_associated_gas_fuel_oil', 'ef_fgtv_distribution_tonne_ch4_per_m3_fuel_crude', 'year'}


In [13]:
all_fields = matt.all_variable_fields_input
df_fiels = df_inputs_raw.columns.tolist()
missing_fields = list(set(all_fields) - set(df_fiels))
print("Missing fields in the input data frame:")
print(missing_fields)

Missing fields in the input data frame:
['frac_lndu_seasonal_wetland_pastures', 'frac_lndu_seasonal_wetland_shrublands', 'frac_frst_forest_conversions_available_for_fuelwood', 'frac_lndu_seasonal_wetland_croplands', 'frac_lndu_seasonal_wetland_other', 'frac_trns_fuelmix_water_borne_gasoline', 'frac_lndu_seasonal_wetland_grasslands']


In [14]:
# Ensure if time_period field exist
if 'time_period' not in df_inputs_raw.columns:
    logger.info("Adding 'time_period' column to df_inputs_raw")
    df_inputs_raw = df_inputs_raw.rename(columns={'period':'time_period'})
else:
    logger.info("'time_period' column already exists in df_inputs_raw")

2026-09-17 16:43:09,759 - INFO - 'time_period' column already exists in df_inputs_raw


In [15]:
# Fixes differences and makes sure that our df is in the correct format.
# Note: Edit this if you need more changes in your df

df_inputs_raw_complete = g_utils.add_missing_cols(df_example_input, df_inputs_raw.copy())
df_inputs_raw_complete.head()

,time_period,year,area_gnrl_country_ha,area_lndu_infimum_croplands_ha,area_lndu_infimum_flooded_ha,area_lndu_infimum_forests_mangroves_ha,area_lndu_infimum_forests_primary_ha,area_lndu_infimum_forests_secondary_ha,area_lndu_infimum_grasslands_ha,area_lndu_infimum_other_ha,...,yf_agrc_vegetables_and_vines_tonne_ha,yf_lndu_supremum_pastures_tonne_per_ha,region,frac_trns_fuelmix_water_borne_gasoline,frac_frst_forest_conversions_available_for_fuelwood,frac_lndu_seasonal_wetland_croplands,frac_lndu_seasonal_wetland_grasslands,frac_lndu_seasonal_wetland_other,frac_lndu_seasonal_wetland_pastures,frac_lndu_seasonal_wetland_shrublands
0,0,2015,44655000,-999,-999,-999,-999,-999,-999,-999,...,24.760000,92.81,costa_rica,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,1,2016,44655000,-999,-999,-999,-999,-999,-999,-999,...,25.570737,92.81,costa_rica,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,2,2017,44655000,-999,-999,-999,-999,-999,-999,-999,...,24.673359,92.81,costa_rica,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,3,2018,44655000,-999,-999,-999,-999,-999,-999,-999,...,25.327768,92.81,costa_rica,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,4,2019,44655000,-999,-999,-999,-999,-999,-999,-999,...,28.151940,92.81,costa_rica,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [16]:
# Double checking that our df is in the correct shape (Empty sets should be printed to make sure everything is Ok!)
g_utils.compare_dfs(df_example_input, df_inputs_raw_complete)

Columns in df_example but not in df_input: set()
Columns in df_input but not in df_example: {'frac_fgtv_capture_associated_gas_fuel_crude', 'ef_fgtv_distribution_tonne_co2_per_m3_fuel_crude', 'frac_fgtv_capture_associated_gas_fuel_coal', 'iso_alpha_3', 'frac_fgtv_capture_associated_gas_fuel_oil', 'ef_fgtv_distribution_tonne_ch4_per_m3_fuel_crude', 'year'}


In [17]:
df_inputs_raw_complete["region"].head()

0    costa_rica
1    costa_rica
2    costa_rica
3    costa_rica
4    costa_rica
Name: region, dtype: object

In [18]:
# Set region to country name
df_inputs_raw_complete['region'] = country_name
df_inputs_raw_complete['region'].head()

0    morocco
1    morocco
2    morocco
3    morocco
4    morocco
Name: region, dtype: object

In [19]:
# filter to match sim_end_year
print(f"min and max years in raw inputs before filtering: {df_inputs_raw_complete['year'].min()} to {df_inputs_raw_complete['year'].max()}")
df_inputs_raw_complete = df_inputs_raw_complete[df_inputs_raw_complete['year'] <= sim_end_year]
print(f"min and max years in raw inputs after filtering: {df_inputs_raw_complete['year'].min()} to {df_inputs_raw_complete['year'].max()}")

min and max years in raw inputs before filtering: 2015 to 2070
min and max years in raw inputs after filtering: 2015 to 2050


## Let's Modify the  LNDU Reallocation factor

In [20]:
if set_lndu_reallocation_factor_to_zero_flag:
    df_inputs_raw_complete['lndu_reallocation_factor'] = 0

df_inputs_raw_complete['lndu_reallocation_factor'].mean()

np.float64(0.0)

In [21]:
path_cur = pathlib.Path(os.getcwd())
path_transformations = path_cur.parent.joinpath("transformations")
path_transformations

PosixPath('/Users/fabianfuentes/git/ssp_morocco/ssp_modeling/transformations')

In [22]:
import sisepuede.transformers as trf
import sisepuede.transformers.transformations as tmts
importlib.reload(tmts.trs)


transformers = trf.Transformers(
    {},
    df_input = df_inputs_raw_complete,
)


# initialize a transformations set
if not path_transformations.is_dir():
    trf.instantiate_default_strategy_directory(
        transformers,
        path_transformations,
    )

In [23]:
transformations = tmts.Transformations(
    path_transformations,
    df_input = df_inputs_raw_complete,
)

In [24]:
transformers = transformations.transformers

strategies = trf.Strategies(
    transformations,
    export_path = "transformations",
    prebuild = True,
)

In [25]:
strategies.attribute_table.table

,strategy_id,strategy_code,strategy,description,transformation_specification,baseline_strategy_id
0,0,BASE,Strategy TX:BASE,NaN,TX:BASE,1
1,1000,AGRC:DEC_CH4_RICE,Singleton - Default Value - AGRC: Improve rice...,NaN,TX:AGRC:DEC_CH4_RICE,0
2,1001,AGRC:DEC_EXPORTS,Singleton - Default Value - AGRC: Decrease Exp...,NaN,TX:AGRC:DEC_EXPORTS,0
3,1002,AGRC:DEC_LOSSES_SUPPLY_CHAIN,Singleton - Default Value - AGRC: Reduce suppl...,NaN,TX:AGRC:DEC_LOSSES_SUPPLY_CHAIN,0
4,1003,AGRC:INC_CONSERVATION_AGRICULTURE,Singleton - Default Value - AGRC: Expand conse...,NaN,TX:AGRC:INC_CONSERVATION_AGRICULTURE,0
...,...,...,...,...,...,...
69,6000,PFLO:INC_HEALTHIER_DIETS,Singleton - Default Value - PFLO: Change diets,NaN,TX:PFLO:INC_HEALTHIER_DIETS,0
70,6001,PFLO:INC_IND_CCS,Singleton - Default Value - PFLO: Industrial c...,NaN,TX:PFLO:INC_IND_CCS,0
71,6002,PFLO:ALL,All Actions,All actions (unique by transformer),TX:AGRC:DEC_CH4_RICE|TX:AGRC:DEC_EXPORTS|TX:AG...,0
72,6003,PFLO:BAU,bau,bau,TX:ENTC:TARGET_CLEAN_HYDROGEN_STRATEGY_BAU|TX:...,0


In [26]:

attribute_strategy_new = sf._concat_df(
    [
        strategies.attribute_table.table,
        strategies.build_whirlpool_strategies(6004, )
    ]
)

attribute_strategy_new

,strategy_id,strategy_code,strategy,description,transformation_specification,baseline_strategy_id
0,0,BASE,Strategy TX:BASE,NaN,TX:BASE,1.0
1,1000,AGRC:DEC_CH4_RICE,Singleton - Default Value - AGRC: Improve rice...,NaN,TX:AGRC:DEC_CH4_RICE,0.0
2,1001,AGRC:DEC_EXPORTS,Singleton - Default Value - AGRC: Decrease Exp...,NaN,TX:AGRC:DEC_EXPORTS,0.0
3,1002,AGRC:DEC_LOSSES_SUPPLY_CHAIN,Singleton - Default Value - AGRC: Reduce suppl...,NaN,TX:AGRC:DEC_LOSSES_SUPPLY_CHAIN,0.0
4,1003,AGRC:INC_CONSERVATION_AGRICULTURE,Singleton - Default Value - AGRC: Expand conse...,NaN,TX:AGRC:INC_CONSERVATION_AGRICULTURE,0.0
...,...,...,...,...,...,...
111,6042,WHIRLPOOL_PFLO_LEDS:TX:TRWW:INC_CAPTURE_BIOGAS...,Remove TX:TRWW:INC_CAPTURE_BIOGAS_STRATEGY_LED...,,TX:AGRC:DEC_EXPORTS_STRATEGY_LEDS|TX:CCSQ:INC_...,0.0
112,6043,WHIRLPOOL_PFLO_LEDS:TX:TRWW:INC_COMPLIANCE_SEP...,Remove TX:TRWW:INC_COMPLIANCE_SEPTIC_STRATEGY_...,,TX:AGRC:DEC_EXPORTS_STRATEGY_LEDS|TX:CCSQ:INC_...,0.0
113,6044,WHIRLPOOL_PFLO_LEDS:TX:WALI:INC_TREATMENT_INDU...,Remove TX:WALI:INC_TREATMENT_INDUSTRIAL_STRATE...,,TX:AGRC:DEC_EXPORTS_STRATEGY_LEDS|TX:CCSQ:INC_...,0.0
114,6045,WHIRLPOOL_PFLO_LEDS:TX:WASO:DEC_CONSUMER_FOOD_...,Remove TX:WASO:DEC_CONSUMER_FOOD_WASTE_STRATEG...,,TX:AGRC:DEC_EXPORTS_STRATEGY_LEDS|TX:CCSQ:INC_...,0.0


In [27]:
attribute_strategy_new.to_csv(
    path_transformations.joinpath("strategy_definition_whirlpool.csv"),
    encoding = "UTF-8",
    index = None
)

In [28]:

attribute_strategy_new = sf._concat_df(
    [
        strategies.attribute_table.table,
       strategies.build_tornado_strategies(strategy_base=None,     
                                           strategy_stress=6004,)
    ]
)

attribute_strategy_new

,strategy_id,strategy_code,strategy,description,transformation_specification,baseline_strategy_id
0,0,BASE,Strategy TX:BASE,NaN,TX:BASE,1.0
1,1000,AGRC:DEC_CH4_RICE,Singleton - Default Value - AGRC: Improve rice...,NaN,TX:AGRC:DEC_CH4_RICE,0.0
2,1001,AGRC:DEC_EXPORTS,Singleton - Default Value - AGRC: Decrease Exp...,NaN,TX:AGRC:DEC_EXPORTS,0.0
3,1002,AGRC:DEC_LOSSES_SUPPLY_CHAIN,Singleton - Default Value - AGRC: Reduce suppl...,NaN,TX:AGRC:DEC_LOSSES_SUPPLY_CHAIN,0.0
4,1003,AGRC:INC_CONSERVATION_AGRICULTURE,Singleton - Default Value - AGRC: Expand conse...,NaN,TX:AGRC:INC_CONSERVATION_AGRICULTURE,0.0
...,...,...,...,...,...,...
111,6042,TORNADO_BASE:TX:TRWW:INC_CAPTURE_BIOGAS_STRATE...,Add TX:TRWW:INC_CAPTURE_BIOGAS_STRATEGY_LEDS t...,,TX:TRWW:INC_CAPTURE_BIOGAS_STRATEGY_LEDS,0.0
112,6043,TORNADO_BASE:TX:TRWW:INC_COMPLIANCE_SEPTIC_STR...,Add TX:TRWW:INC_COMPLIANCE_SEPTIC_STRATEGY_LED...,,TX:TRWW:INC_COMPLIANCE_SEPTIC_STRATEGY_LEDS,0.0
113,6044,TORNADO_BASE:TX:WALI:INC_TREATMENT_INDUSTRIAL_...,Add TX:WALI:INC_TREATMENT_INDUSTRIAL_STRATEGY_...,,TX:WALI:INC_TREATMENT_INDUSTRIAL_STRATEGY_LEDS,0.0
114,6045,TORNADO_BASE:TX:WASO:DEC_CONSUMER_FOOD_WASTE_S...,Add TX:WASO:DEC_CONSUMER_FOOD_WASTE_STRATEGY_L...,,TX:WASO:DEC_CONSUMER_FOOD_WASTE_STRATEGY_LEDS,0.0


In [29]:
attribute_strategy_new.to_csv(
    path_transformations.joinpath("strategy_definition_tornado.csv"),
    encoding = "UTF-8",
    index = None
)

In [30]:
matt.get_dimensional_attribute_table(matt.dim_time_period).table

,time_period,year
0,0,2015
1,1,2016
2,2,2017
3,3,2018
4,4,2019
5,5,2020
6,6,2021
7,7,2022
8,8,2023
9,9,2024


In [31]:

##  WRITE SOME OBJECTS FOR USE WITH CLOUD

sf._write_csv(
    matt.get_dimensional_attribute_table(matt.dim_time_period).table,
    path_transformations.joinpath("attribute_dim_time_period.csv")
)

True

In [32]:
strategies.get_strategy(6004).code

'PFLO:LEDS'